In [33]:
import pandas as pd 

## Cargar datos

In [34]:
posts_file = "data/rspct_autos.tsv.gz"
posts_df = pd.read_csv(posts_file, sep='\t')

In [35]:
subred_file = "data/subreddit_info.csv.gz"
subred_df = pd.read_csv(subred_file).set_index(['subreddit'])

df = posts_df.join(subred_df, on='subreddit')


In [36]:
df.head()

,id,subreddit,title,selftext,category_1,category_2,category_3,in_data,reason_for_exclusion
0,8f73s7,Harley,No Club Colors,Funny story. I went to college in Las Vegas. T...,autos,harley davidson,NaN,True,NaN
1,5s0q8r,Mustang,Roush vs Shleby GT500,"I am trying to determine which is faster, and ...",autos,ford,NaN,True,NaN
2,5z3405,Volkswagen,2001 Golf Wagon looking for some insight,Hello! <lb><lb>Trying to find some information...,autos,VW,NaN,True,NaN
3,7df18v,Lexus,IS 250 Coolant Flush/Change,https://www.cars.com/articles/how-often-should...,autos,lexus,NaN,True,NaN
4,5tpve8,volt,Gen1 mpg w/ dead battery?,"Hi, new to this subreddit. I'm considering bu...",autos,chevrolet,NaN,True,NaN


## Blueprint: Estandarizar nombres de atributos

In [37]:
print(df.columns)

Index(['id', 'subreddit', 'title', 'selftext', 'category_1', 'category_2',
       'category_3', 'in_data', 'reason_for_exclusion'],
      dtype='object')


In [38]:
column_mapping = {
    'id' : 'id',
    'subreddit' : 'subreddit',
    'title' : 'title',
    'selftext' : 'text',
    'category_1' : 'category',
    'category_2' : 'subcategory',
    'category_3' : None, #no data 
    'in_data' : None, #not needed
    'reason_for_exlusion': None #not needed
}

In [39]:
columns = [c for c in column_mapping.keys() if column_mapping[c] != None]

In [40]:
df = df[columns].rename(columns=column_mapping)

In [41]:
df = df[df['category'] == 'autos']
df.sample(1).T

,1807
id,7zoy8u
subreddit,Porsche
title,How to Clean Porsche?
text,"Hey everyone,<lb><lb>Living in the northeast, ..."
category,autos
subcategory,porsche


### Guardar y cargar datos a un DataFrame



In [42]:
import sqlite3

db_name = "reddit_selfpost.db"
con = sqlite3.connect(db_name)
df.to_sql("posts", con, index=False, if_exists="replace")
con.close()

In [43]:
con = sqlite3.connect(db_name)
df = pd.read_sql("select * from posts", con)
con.close()

## Cleaning text data

In [44]:
text ="""
After viewing the [PINKIEPOOL Trailer](https://www.youtu.be/watch?v=ieHRoHUg)
it got me thinking about the best match ups.
<lb>Here's my take:<lb><lb>[](/sp)[](/ppseesyou) Deadpool<lb>[](/sp)[](/ajsly)
Captain America<lb>"""

### Blueprint: Identificación de ruido con regex


In [45]:
import re

RE_SUSPICIUS = re.compile(r'[&#<>{}\[\]\\]')

def impurity(text, min_len=10):
    if text == None or len(text)< min_len:
        return 0
    else:
        return len(RE_SUSPICIUS.findall(text))/len(text)
    
print(impurity(text))

0.09009009009009009


El 9% de los caracteres son suspechosos de acuerdo con la definición de textos redactados correctamente. El patrón de búsqueda puede necesitar adaptaciones para corpora que contiene hastags o token similares que contenga caracteres especiales.

Sin embargo, no necesita ser perfecto, solo nocesitan ser lo suficientemente bueno para indicar una calidad potencial.



In [46]:
df['impurity'] = df['text'].apply(impurity, min_len=10)

In [47]:
df[['text', 'impurity']].sort_values(by='impurity', ascending=False).head(3)

,text,impurity
19682,Looking at buying a 335i with 39k miles and 11...,0.214716
12357,I'm looking to lease an a4 premium plus automa...,0.165099
2730,Breakdown below:<lb><lb>Elantra GT<lb><lb>2.0L...,0.139130


In [48]:
from collections import Counter

def count_words(df, column='tokens', preprocess=None, min_freq=2):

    # process tokens and update counter
    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(tokens)

    # create counter and run through all data
    counter = Counter()
    df[column].progress_map(update)

    # transform counter into data frame
    freq_df = pd.DataFrame.from_dict(counter, orient='index', columns=['freq'])
    freq_df = freq_df.query('freq >= @min_freq')
    freq_df.index.name = 'token'
    return freq_df.sort_values('freq', ascending=False)

In [49]:
import pandas as pd
from tqdm import tqdm
from collections import Counter

tqdm.pandas()  # Habilita barra de progreso en pandas

def count_words(df, column='tokens', preprocess=None, min_freq=2):

    # process tokens and update counter
    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(tokens)

    # create counter and run through all data
    counter = Counter()
    df[column].progress_apply(update)  # <--- corrección aquí

    # transform counter into data frame
    freq_df = pd.DataFrame.from_dict(counter, orient='index', columns=['freq'])
    freq_df = freq_df.query('freq >= @min_freq')
    freq_df.index.name = 'token'
    return freq_df.sort_values('freq', ascending=False)

In [50]:
count_words(df, column='text', preprocess=lambda t : re.findall(r'<[\w/]*>', t))

100%|██████████| 20000/20000 [00:00<00:00, 487528.36it/s]


,freq
token,
<lb>,100729
<tab>,642


### Blueprint: Remover ruido con regex



In [51]:
import html

def clean(text):
    # convert html escapes like &amp; to characters.
    text = html.unescape(text)
    # tags like <tab>
    text = re.sub(r'<[^<>]*>', ' ', text)
    # markdown URLs like [Some text](https://....)
    text = re.sub(r'\[([^\[\]]*)\]\([^\(\)]*\)', r'\1', text)
    # text or code in brackets like [0]
    text = re.sub(r'\[[^\[\]]*\]', ' ', text)
    # standalone sequences of specials, matches &# but not #cool
    text = re.sub(r'(?:^|\s)[&#<>{}\[\]+|\\:-]{1,}(?:\s|$)', ' ', text)
    # standalone sequences of hyphens like --- or ==
    text = re.sub(r'(?:^|\s)[\-=\+]{2,}(?:\s|$)', ' ', text)
    # sequences of white spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [52]:
clean_text = clean(text)
print(clean_text)
print("Impurity:", impurity(clean_text))

After viewing the PINKIEPOOL Trailer it got me thinking about the best match ups. Here's my take: Deadpool Captain America
Impurity: 0.0


In [53]:
df['clean_text'] = df['text'].map(clean)
df['impurity'] = df['clean_text'].apply(impurity, min_len=20)
df[['clean_text', 'impurity']].sort_values(by='impurity', ascending=False).head(3)

,clean_text,impurity
14058,"Mustang 2018, 2019, or 2020? Must Haves!! 1. H...",0.030864
18934,"At the dealership, they offered an option for ...",0.026455
16505,"I am looking at four Caymans, all are in a sim...",0.024631


### Blueprint: Normalización de caracteres con textacy



In [54]:
text = "The café “Saint-Raphaël” is loca-\nted on Côte dʼAzur."

In [55]:
import textacy.preprocessing as tprep

In [56]:
def normalize_hyphenated_words(text):
    # Une palabras divididas por guiones y salto de línea
    return re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)

In [57]:
def normalize_quotation_marks(text):
    # Reemplaza comillas tipográficas por comillas rectas
    return text.translate(str.maketrans({
        '“': '"', '”': '"',
        '‘': "'", '’': "'"
    }))

In [58]:
import unicodedata

def normalize_unicode(text, form='NFKC'):
    return unicodedata.normalize(form, text)


In [59]:
import unicodedata

def remove_accents(text):
    return ''.join(
        char for char in unicodedata.normalize('NFD', text)
        if unicodedata.category(char) != 'Mn'
    )


In [60]:
def normalize(text):
    #text = tprep.normalize_hyphenated_words(text)
    text = normalize_hyphenated_words(text)
    #text = tprep.normalize_quotation_marks(text)
    text = normalize_quotation_marks(text)
    #text = tprep.normalize_unicode(text)
    text = normalize_unicode(text)
    #text = tprep.remove_accents(text)
    text = remove_accents(text)
    return text

print(normalize(text))


The cafe "Saint-Raphael" is located on Cote dʼAzur.


# Blueprint: Enmascarar datos con textacy basado en patrones

In [61]:
from textacy.preprocessing.resources import RE_URL
count_words(df, column='clean_text', preprocess=RE_URL.findall).head(3)

100%|██████████| 20000/20000 [00:00<00:00, 37385.99it/s]


,freq
token,
www.getlowered.com,3
http://www.ecolamautomotive.com/#!2/kv7fq,2
https://www.reddit.com/r/Jeep/comments/4ux232/just_ordered_an_android_head_unit_joying_jeep/,2


In [65]:
from textacy.preprocessing import replace_urls




ImportError: cannot import name 'replace_urls' from 'textacy.preprocessing' (/opt/miniconda3/envs/albrencht-book/lib/python3.10/site-packages/textacy/preprocessing/__init__.py)

In [67]:
import re

def replace_urls(text, replacement="_URL_"):
    return re.sub(r"http\S+|www\S+|https\S+", replacement, text)

In [68]:
text = "Check out https://spacy.io/usage/spacy-101"
# using default substitution _URL_
print(replace_urls(text))

Check out _URL_


In [69]:
df['clean_text'] = df['clean_text'].map(replace_urls)
df['clean_text'] = df['clean_text'].map(normalize)

In [70]:
df['clean_text']

0        Funny story. I went to college in Las Vegas. T...
1        I am trying to determine which is faster, and ...
2        Hello! Trying to find some information on repl...
3        _URL_ I have a IS 250 AWD from 2006. About 73K...
4        Hi, new to this subreddit. I'm considering buy...
                               ...                        
19995    I read a lot Forums and people recommend getti...
19996    I am thinking about buying a 2010 Harley Sport...
19997    My husband and I were headed somewhere and I w...
19998    I am looking at getting a used Lexus IS (2014 ...
19999    Looking for some help. I've never owned any lu...
Name: clean_text, Length: 20000, dtype: object

In [ ]:
df.rename(columns={'text': 'raw_text', 'clean_text': 'text'}, inplace=True)
df.drop(columns=['impurity'], inplace=True)

con = sqlite3.connect(db_name)
df.to_sql("posts_cleaned", con, index=False, if_exists="replace")
con.close()